<a href="https://colab.research.google.com/github/ai-agent-yonsei/agent/blob/main/4_2_MCP_Multi_Server_w_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 10. 멀티 MCP 서버 실습

### 처리 흐름 7단계 (질문마다 ①~⑦ 배너로 출력)
① 도구 목록 확보(사전 준비, **서버별로** `tools/list`) → ② 질문 수신 → ③ 질문 + 전체 도구 목록 전달 → ④ 도구 호출 판단 → ⑤ MCP 서버 호출(**라우팅**) → ⑥ 도구 실행(외부 시스템) → ⑦ 최종 답변

## 0. 환경 준비

In [14]:
!pip -q install "mcp>=1.10,<2" openai mcp-server-fetch

from importlib.metadata import version
for pkg in ("mcp", "openai", "mcp-server-fetch"):
    print(f"{pkg:18}: {version(pkg)}")

mcp               : 1.30.0
openai            : 2.54.0
mcp-server-fetch  : 2026.8.18


In [15]:
import os, time

# 시간대 설정: Colab 런타임 기본값은 UTC → 서울 시간(KST, UTC+9)으로 변경
TIMEZONE = "Asia/Seoul"
os.environ["TZ"] = TIMEZONE
time.tzset()

def load_secret(name, required=False):
    """Colab Secrets → 환경변수. 없고 필수면 직접 입력받음."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = os.environ.get(name)
    if not value and required:
        from getpass import getpass
        value = getpass(f"{name} 입력: ")
    if value:
        os.environ[name] = value
    print(f"{'[OK]  ' if value else '[없음]'} {name}")
    return bool(value)

load_secret("OPENAI_API_KEY", required=True)
load_secret("TAVILY_API_KEY", required=True)

MODEL = "gpt-5-mini"

[OK]   OPENAI_API_KEY
[OK]   TAVILY_API_KEY


## 1. MCP 서버 만들기


`%%writefile`로 셀을 실행하면 Colab 작업 폴더에 `.py` 파일이 생성

### 1-1. Tavily 웹 검색 서버 — `tavily_server.py`

In [16]:
%%writefile tavily_server.py
# =====================================================================
# MCP 서버: tavily  —  Tavily Search / Extract API를 MCP 도구로 감싼 서버
# ---------------------------------------------------------------------
# • 호스트가 "python tavily_server.py"로 띄우는 별도 프로세스
# • stdin  ← 클라이언트의 JSON-RPC 요청 / stdout → 응답
#   주의: stdout은 프로토콜 전용. 로그는 반드시 stderr로!
# • TAVILY_API_KEY는 호스트가 '이 서버에게만' 환경변수로 전달 (LLM은 모름)
# • @mcp.tool() 함수 = 도구.  함수명→name, docstring→description,
#   타입힌트·Field→inputSchema 로 자동 변환되어 tools/list 응답이 됨
# =====================================================================
import os, sys, json, urllib.request
from datetime import datetime, timedelta, timezone
from typing import Annotated, Literal
from pydantic import Field
from mcp.server.fastmcp import FastMCP

KST = timezone(timedelta(hours=9))
TAVILY_BASE = "https://api.tavily.com"

mcp = FastMCP("tavily-search-server")


def log(msg: str) -> None:
    print(f"[TAVILY] {msg}", file=sys.stderr, flush=True)


def _tavily_post(path: str, payload: dict) -> dict:
    """외부 시스템(Tavily API) 호출. API 키는 헤더로만 전송하고 로그에는 남기지 않음."""
    key = os.environ.get("TAVILY_API_KEY")
    if not key:
        raise RuntimeError("TAVILY_API_KEY가 서버 환경변수에 없습니다.")
    log(f"⑥ 외부 시스템 호출 → POST {TAVILY_BASE}{path}  payload={json.dumps(payload, ensure_ascii=False)}")
    req = urllib.request.Request(
        TAVILY_BASE + path,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {key}"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=40) as r:
        return json.loads(r.read().decode("utf-8"))


@mcp.tool() ## 함수 위에 붙으면 그 함수가 MCP도구로 등록된당
def tavily_search(
    query: Annotated[str, Field(description="검색어. 예: 'AI agent news', 'AI 에이전트'")],
    topic: Annotated[Literal["news", "general"], Field(description="'news'=뉴스 기사 위주, 'general'=일반 웹")] = "news",
    time_range: Annotated[Literal["day", "week", "month"], Field(description="검색 기간(최근 하루/일주일/한 달)")] = "week",
    max_results: Annotated[int, Field(ge=1, le=10, description="반환할 최대 결과 수")] = 8,
    search_depth: Annotated[Literal["basic", "advanced"], Field(description="'advanced'는 더 정확하지만 크레딧을 더 사용")] = "advanced",
) -> dict:
    """Tavily로 웹/뉴스를 검색합니다. 각 결과의 제목, URL, 발행일, 관련도 점수, 본문 발췌를 돌려줍니다.
    최신 뉴스·동향·사실 확인이 필요한 질문에 우선적으로 사용하세요."""
    log(f"⑥ tools/call 수신 → tavily_search(query={query!r}, topic={topic}, time_range={time_range}, max_results={max_results})")
    try:
        data = _tavily_post("/search", {
            "query": query, "topic": topic, "time_range": time_range,
            "max_results": max_results, "search_depth": search_depth,
        })
    except Exception as e:
        log(f"⑥ 외부 시스템 오류: {e!r}")
        return {"ok": False, "error": str(e), "results": []}

    results = [{
        "title": r.get("title"),
        "url": r.get("url"),
        "published_date": r.get("published_date"),
        "score": round(r.get("score", 0), 3),
        "content": (r.get("content") or "")[:700],   # LLM 컨텍스트 절약을 위해 발췌만
    } for r in data.get("results", [])]
    log(f"⑥ 결과 정리 완료 → {len(results)}건 반환 (Tavily 응답시간 {data.get('response_time')}s)")
    return {
        "ok": True,
        "query": query,
        "searched_at": datetime.now(KST).strftime("%Y-%m-%d %H:%M KST"),
        "count": len(results),
        "results": results,
    }


@mcp.tool()
def tavily_extract(
    urls: Annotated[list[str], Field(min_length=1, max_length=3, description="본문을 가져올 URL 목록(최대 3개)")],
) -> dict:
    """URL의 본문 텍스트를 추출합니다. tavily_search의 발췌만으로 요약이 부족할 때 사용하세요."""
    log(f"⑥ tools/call 수신 → tavily_extract({len(urls)}개 URL)")
    try:
        data = _tavily_post("/extract", {"urls": urls})
    except Exception as e:
        log(f"⑥ 외부 시스템 오류: {e!r}")
        return {"ok": False, "error": str(e), "results": []}
    results = [{"url": r.get("url"), "content": (r.get("raw_content") or "")[:3000]}
               for r in data.get("results", [])]
    log(f"⑥ 결과 정리 완료 → 본문 {len(results)}건 반환 (실패 {len(data.get('failed_results', []))}건)")
    return {"ok": True, "count": len(results), "results": results}


if __name__ == "__main__":
    log("서버 프로세스 시작 — stdio로 JSON-RPC 요청 대기 중")
    mcp.run(transport="stdio")

Overwriting tavily_server.py


### 1-2. 날씨 서버 — `weather_server.py` (Open-Meteo, 키 불필요)

도시 이름 → (지오코딩 API) → 위도·경도 → (예보 API) → 현재 날씨 + 일별 예보

In [17]:
%%writefile weather_server.py
# =====================================================================
# MCP 서버: weather  —  Open-Meteo API를 MCP 도구로 감싼 서버
#   (슬라이드의 get_weather 예시를 실제로 동작하게 구현)
# =====================================================================
import sys, json, urllib.parse, urllib.request
from typing import Annotated
from pydantic import Field
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("open-meteo-weather-server")

# WMO 날씨 코드 → 한국어
WMO = {0: "맑음", 1: "대체로 맑음", 2: "구름 조금", 3: "흐림", 45: "안개", 48: "서리 안개",
       51: "약한 이슬비", 53: "이슬비", 55: "강한 이슬비", 61: "약한 비", 63: "비", 65: "강한 비",
       66: "약한 어는 비", 67: "어는 비", 71: "약한 눈", 73: "눈", 75: "강한 눈", 77: "싸락눈",
       80: "약한 소나기", 81: "소나기", 82: "강한 소나기", 85: "약한 눈 소나기", 86: "눈 소나기",
       95: "뇌우", 96: "우박 동반 뇌우", 99: "강한 우박 동반 뇌우"}

# 지오코딩이 한글 지명을 못 찾을 때를 대비한 간단한 별칭
ALIASES = {"서울": "Seoul", "부산": "Busan", "인천": "Incheon", "대구": "Daegu", "대전": "Daejeon",
           "광주": "Gwangju", "울산": "Ulsan", "제주": "Jeju", "수원": "Suwon", "도쿄": "Tokyo", "뉴욕": "New York"}


def log(msg: str) -> None:
    print(f"[WEATHER] {msg}", file=sys.stderr, flush=True)


def _get_json(url: str) -> dict:
    log(f"⑥ 외부 시스템 호출 → GET {url}")
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "MCP-Weather-Demo"}), timeout=20) as r:
        return json.loads(r.read().decode("utf-8"))


def _geocode(city: str) -> dict | None:
    for name in (city, ALIASES.get(city.strip())):
        if not name:
            continue
        q = urllib.parse.urlencode({"name": name, "count": 1, "language": "ko", "format": "json"})
        hits = _get_json(f"https://geocoding-api.open-meteo.com/v1/search?{q}").get("results")
        if hits:
            return hits[0]
    return None


@mcp.tool()
def get_weather(
    city: Annotated[str, Field(description="도시 이름. 예: 'Seoul', '서울', 'Busan'")],
    forecast_days: Annotated[int, Field(ge=1, le=7, description="일별 예보 일수")] = 3,
) -> dict:
    """도시의 현재 날씨(기온, 체감온도, 습도, 강수, 풍속, 날씨 상태)와 일별 예보(최고/최저 기온, 강수 확률)를 조회합니다.
    날씨·기온·비 소식 등을 묻는 질문에 사용하세요."""
    log(f"⑥ tools/call 수신 → get_weather(city={city!r}, forecast_days={forecast_days})")
    try:
        place = _geocode(city)
        if not place:
            return {"ok": False, "error": f"'{city}' 위치를 찾지 못했습니다. 영문 도시명으로 다시 시도하세요."}
        log(f"⑥ 지오코딩 결과 → {place['name']} ({place['latitude']:.2f}, {place['longitude']:.2f})")
        q = urllib.parse.urlencode({
            "latitude": place["latitude"], "longitude": place["longitude"], "timezone": "auto",
            "forecast_days": forecast_days,
            "current": "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,weather_code,wind_speed_10m",
            "daily": "weather_code,temperature_2m_max,temperature_2m_min,precipitation_probability_max",
        })
        data = _get_json(f"https://api.open-meteo.com/v1/forecast?{q}")
    except Exception as e:
        log(f"⑥ 외부 시스템 오류: {e!r}")
        return {"ok": False, "error": str(e)}

    cur, daily = data["current"], data["daily"]
    result = {
        "ok": True,
        "location": f"{place['name']}, {place.get('admin1', '')} {place.get('country', '')}".strip(),
        "timezone": data.get("timezone"),
        "current": {
            "time": cur["time"],
            "condition": WMO.get(cur["weather_code"], f"코드 {cur['weather_code']}"),
            "temperature_c": cur["temperature_2m"],
            "feels_like_c": cur["apparent_temperature"],
            "humidity_pct": cur["relative_humidity_2m"],
            "precipitation_mm": cur["precipitation"],
            "wind_kmh": cur["wind_speed_10m"],
        },
        "daily": [{
            "date": daily["time"][i],
            "condition": WMO.get(daily["weather_code"][i], ""),
            "max_c": daily["temperature_2m_max"][i],
            "min_c": daily["temperature_2m_min"][i],
            "rain_prob_pct": daily["precipitation_probability_max"][i],
        } for i in range(len(daily["time"]))],
    }
    log(f"⑥ 결과 정리 완료 → 현재 {result['current']['temperature_c']}°C, {result['current']['condition']}")
    return result


if __name__ == "__main__":
    log("서버 프로세스 시작 — stdio로 JSON-RPC 요청 대기 중")
    mcp.run(transport="stdio")

Overwriting weather_server.py


### 1-3. 공개 MCP 서버 — `mcp-server-fetch`

## 2. MCP 서버 설정 — `MCP_SERVERS`  

Claude Desktop·Cursor 등 MCP 호스트 앱의 `mcpServers` 설정과 같은 모양  **서버를 추가하려면 이 딕셔너리에 블록 하나만 추가**
- `command` + `args` : 서버를 띄우는 명령어 (stdio 방식의 '연결 정보'는 URL이 아니라 **실행 명령** )
- `env` : 그 서버에게만 전달할 환경변수(API 키 등)
- `enabled` : `False`로 두면 연결하지 않음

In [18]:
import sys, os

MCP_SERVERS = {
    "tavily": {
        "description": "Tavily 웹/뉴스 검색 (직접 구현)",
        "command": sys.executable,
        "args": ["tavily_server.py"],
        "env": {"TAVILY_API_KEY": os.environ.get("TAVILY_API_KEY", "")},   # 이 서버만 Tavily 키를 받음
        "enabled": True,
    },
    "weather": {
        "description": "Open-Meteo 날씨 조회 (직접 구현, 키 불필요)",
        "command": sys.executable,
        "args": ["weather_server.py"],
        "env": {},
        "enabled": True,
    },
    "fetch": {
        "description": "웹페이지 → 마크다운 (공개 레퍼런스 서버 mcp-server-fetch)",
        "command": sys.executable,
        "args": ["-m", "mcp_server_fetch"],
        "env": {},
        "enabled": True,
    },

    # ───────── 서버 추가 예시 (주석 해제 후 사용) ─────────
    # (1) 내가 만든 파이썬 서버
    # "my_tools": {
    #     "description": "내가 만든 도구 서버",
    #     "command": sys.executable,
    #     "args": ["my_tools_server.py"],
    #     "env": {"MY_API_KEY": os.environ.get("MY_API_KEY", "")},
    #     "enabled": True,
    # },
    # (2) Node.js 기반 공개 서버 (Colab에는 node/npx가 설치되어 있음)
    # "memory": {
    #     "description": "지식 그래프 메모리 (공개 레퍼런스 서버)",
    #     "command": "npx",
    #     "args": ["-y", "@modelcontextprotocol/server-memory"],
    #     "env": {},
    #     "enabled": True,
    # },
}

for name, cfg in MCP_SERVERS.items():
    flag = "[ON] " if cfg["enabled"] else "[OFF]"
    print(f"{flag} {name:8} | {cfg['description']:45} | 실행: {os.path.basename(cfg['command'])} {' '.join(cfg['args'])}")

[ON]  tavily   | Tavily 웹/뉴스 검색 (직접 구현)                        | 실행: python3 tavily_server.py
[ON]  weather  | Open-Meteo 날씨 조회 (직접 구현, 키 불필요)               | 실행: python3 weather_server.py
[ON]  fetch    | 웹페이지 → 마크다운 (공개 레퍼런스 서버 mcp-server-fetch)     | 실행: python3 -m mcp_server_fetch


## 3. MCP 호스트 구성요소

### 3-1. 출력 — ①~⑦ 배너, 실행 기록, 서버 로그 보기

In [19]:
import json, time, textwrap
from IPython.display import display, Markdown

STEP_INFO = {
    1: ("도구 목록 확보 (사전 준비)",   "MCP 클라이언트들 ⇄ 각 MCP 서버"),
    2: ("질문 수신",                    "사용자 → 하네스"),
    3: ("질문 + 도구 목록 전달",         "하네스 → LLM"),
    4: ("도구 호출 판단",                "LLM"),
    5: ("MCP 서버 호출 (라우팅)",        "하네스 → 해당 MCP 클라이언트 → 해당 MCP 서버"),
    6: ("도구 실행 (외부 시스템 호출)",   "MCP 서버 → 외부 시스템 → MCP 클라이언트"),
    7: ("최종 답변",                     "하네스 → LLM → 사용자"),
}
CIRCLE = dict(zip(range(1, 8), "①②③④⑤⑥⑦"))
TRACE = []

def banner(step, sub=""):
    title, actor = STEP_INFO[step]
    bar = "═" * 84
    print(f"\n{bar}\n{CIRCLE[step]} [{step}단계] {title}" + (f"  —  {sub}" if sub else ""))
    print(f"   주체: {actor}\n{bar}")

def trace(step, detail):
    TRACE.append((step, detail))

def indent(text, n=6):
    return textwrap.indent(str(text), " " * n)

def clip(text, limit=1500):
    text = str(text)
    return text if len(text) <= limit else text[:limit] + f"\n... (총 {len(text):,}자 중 {limit:,}자만 표시)"

def show_jsonrpc(server, method, params=None):
    """SDK가 해당 서버의 stdin으로 보내는 JSON-RPC 메시지 형태(설명용)"""
    msg = {"jsonrpc": "2.0", "id": "…", "method": method}
    if params is not None:
        msg["params"] = params
    print(f"   → [{server}] JSON-RPC 요청:")
    print(indent(json.dumps(msg, ensure_ascii=False, indent=2), 8))


class ServerLog:
    """서버 프로세스의 stderr 로그 파일에서 새로 추가된 줄만 출력"""
    def __init__(self, path):
        self.path, self.pos = path, 0

    def show_new(self):
        time.sleep(0.05)
        try:
            with open(self.path, encoding="utf-8", errors="replace") as f:
                f.seek(self.pos)
                new = f.read()
                self.pos = f.tell()
        except FileNotFoundError:
            return
        for line in new.splitlines():
            line = line.strip()
            if line.startswith("[") or "Processing request" in line or "Error" in line:
                print("   ", line[:300])

print("[OK] 출력 도우미 준비 완료")

[OK] 출력 도우미 준비 완료


### 3-1b. 감사 로그 (AuditLog)

감사(Audit)의 핵심은 **질문 하나가 처리되는 ①~⑦ 전 과정을, 하나의 추적 ID(trace_id)로 묶어
타임스탬프 순서대로 남기는 것**  


In [20]:
import hashlib, uuid, datetime
from zoneinfo import ZoneInfo

def sha(x, n=12):
    """민감할 수 있는 값은 원문 대신 짧은 해시로 남긴다."""
    return hashlib.sha256(str(x).encode("utf-8")).hexdigest()[:n]

class AuditLog:
    """질문 1건(trace_id) 동안 일어난 ①~⑦ 사건을 타임스탬프 순서로 기록하는 감사 로그."""
    def __init__(self):
        self.records = []        # 전체 누적 (여러 질문)
        self.trace_id = None
        self.seq = 0

    def start(self, question, model, servers):
        """② 직전: 새 질문마다 추적 ID를 발급하고 세션 메타를 남긴다."""
        self.trace_id = uuid.uuid4().hex[:8]
        self.seq = 0
        self.record(0, "호스트", "session_start",
                    model=model, servers=list(servers),
                    question_hash=sha(question), question_len=len(question))
        return self.trace_id

    def record(self, step, actor, event, **fields):
        self.seq += 1
        self.records.append({
            "seq": self.seq,
            "ts": datetime.datetime.now(ZoneInfo(TIMEZONE)).isoformat(timespec="milliseconds"),  # 예: 2026-10-01T17:17:03.123+09:00
            "trace_id": self.trace_id,
            "step": step, "actor": actor, "event": event,
            **fields,
        })

    def table(self, trace_id=None):
        """해당 trace_id의 ①~⑦ 타임라인을 표로 출력 (증거 사슬)."""
        tid = trace_id or self.trace_id
        rows = [r for r in self.records if r["trace_id"] == tid]
        print(f"\n감사 로그 (trace_id={tid}) — ①~⑦ 증거 사슬   [시간대: {TIMEZONE}]")
        print("─" * 112)
        print(f"{'seq':>3} {'time':12} {'단계':>4} {'주체':8} {'event':16} 상세")
        print("─" * 112)
        for r in rows:
            core = {k: v for k, v in r.items()
                    if k not in ("seq","ts","trace_id","step","actor","event")}
            detail = "  ".join(f"{k}={v}" for k, v in core.items())
            t = r["ts"].split("T")[1][:12]    # 표에는 시:분:초.밀리초만
            star = r["step"] if r["step"] else "-"
            print(f"{r['seq']:>3} {t:12} {str(star):>4} {r['actor']:8} {r['event']:16} {detail[:60]}")
        print("─" * 112)
        # ④와 ⑤를 tool_call_id로 이을 수 있는지 검증
        req = {r["tool_call_id"] for r in rows if r["event"]=="llm_tool_request"}
        exe = {r["tool_call_id"] for r in rows if r["event"]=="host_tool_exec"}
        if req == exe and req:
            print(f"[OK] ④요청 ↔ ⑤실행 일치: tool_call_id {sorted(req)} — LLM 요청대로만 실행됨")
        elif req or exe:
            print(f"[경고] 불일치! 요청={sorted(req)} 실행={sorted(exe)} — 요청과 실행이 다름")

AUDIT = AuditLog()
print("[OK] 감사 로그(AuditLog) 준비 완료")

[OK] 감사 로그(AuditLog) 준비 완료


### 3-2. MCP 클라이언트 (서버 1개당 1개)

In [21]:
import anyio
from mcp import ClientSession


class MCPClient:
    """하나의 MCP 서버와 1:1로 연결되는 클라이언트 (tools/list & tools/call 담당)"""

    def __init__(self, name: str, session: ClientSession, server_log: ServerLog):
        self.name = name
        self.session = session
        self.log = server_log
        self.tools = []

    async def connect(self):
        with anyio.fail_after(60):                      # 서버가 응답하지 않으면 60초 후 포기
            init = await self.session.initialize()      # initialize → notifications/initialized
        self.log.show_new()
        print(f"   [{self.name}] 핸드셰이크 완료 — serverInfo='{init.serverInfo.name}', "
              f"protocol={init.protocolVersion}, tools={'지원' if init.capabilities.tools else '미지원'}")

    async def list_tools(self):
        show_jsonrpc(self.name, "tools/list")
        result = await self.session.list_tools()
        self.log.show_new()
        self.tools = result.tools
        print(f"   ← [{self.name}] 도구 {len(self.tools)}개 수신")
        for t in self.tools:
            desc = " ".join(t.description.split())
            params = ", ".join(t.inputSchema.get("properties", {}).keys()) or "-"
            print(f"      {t.name}({params})")
            print(f"          {desc[:150]}{'…' if len(desc) > 150 else ''}")
        return self.tools

    async def call_tool(self, tool_name: str, arguments: dict):
        show_jsonrpc(self.name, "tools/call", {"name": tool_name, "arguments": arguments})
        t0 = time.time()
        result = await self.session.call_tool(tool_name, arguments)
        elapsed = time.time() - t0
        text = "\n".join(c.text for c in result.content if getattr(c, "type", "") == "text")
        return text, bool(result.isError), elapsed

print("[OK] MCPClient 정의 완료")

[OK] MCPClient 정의 완료


### 3-3. LLM (OpenAI)

LLM은 MCP도 서버가 몇 개인지도 모름. 하네스가 건넨 **도구 설명서 목록**만 보고 무엇을 호출할지 판단

In [22]:
from openai import AsyncOpenAI

SYSTEM_PROMPT = """당신은 여러 도구를 활용하는 리서치 어시스턴트입니다.

[공통 규칙]
- 최신 정보나 실시간 정보(뉴스, 날씨 등)는 추측하지 말고 반드시 도구로 확인한 뒤, 도구 결과에 근거해 답하세요.
- 질문에 맞는 도구만 사용하세요. 불필요한 도구는 호출하지 마세요.
- 답변은 한국어 Markdown으로 작성하세요.

[뉴스 질문]
- 뉴스 검색 도구로 국내와 해외(영문)를 모두 살펴보세요.
- 중요도 기준: (1) 산업·시장 파급력 (2) 주요 기업·정책과의 관련성 (3) 여러 매체의 동시 보도. 같은 사건은 하나로 묶으세요.
- 형식: 기준 시각을 먼저 밝히고, 각 뉴스마다 '### 번호. 제목', 출처/날짜, 요약(3~4문장), 왜 중요한가(1문장), 링크.

[날씨 질문]
- 현재 날씨와 오늘·내일 예보를 간결하게 정리하고, 옷차림이나 우산 등 실용적인 한 줄 조언을 덧붙이세요.
"""


class LLM:
    """OpenAI Chat Completions API 래퍼. 하네스만 이 객체를 사용한다.
    ※ LLM이 OpenAI로 바뀌어도 MCP 서버·클라이언트 코드는 전혀 바뀌지 않는다는 점에 주목!
       (MCP는 'LLM과 도구 사이의 표준'이 아니라 '호스트와 도구 서버 사이의 표준'이기 때문)"""

    def __init__(self, model: str = MODEL):
        self.client = AsyncOpenAI()                  # OPENAI_API_KEY 환경변수 사용
        self.model = model

    async def generate(self, messages, tools):
        # OpenAI는 system 프롬프트를 messages의 첫 메시지(role="system")로 전달
        return await self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, *messages],
            tools=tools,                 # ← ③에서 전달되는 '도구 목록'
            tool_choice="auto",          # 도구를 쓸지 말지는 LLM이 판단
            max_completion_tokens=6000,
        )

print("[OK] LLM 정의 완료 (model =", MODEL, ")")

[OK] LLM 정의 완료 (model = gpt-5-mini )


### 3-4. MCP 호스트 — 여러 서버 연결 + 도구 라우팅 테이블

`MCP_SERVERS`를 순회하며 서버마다 ① 프로세스 실행 → ② 클라이언트 연결 → ③ `tools/list`를 수행하고, 모든 도구를 하나의 **라우팅 테이블**에 모음.
한 서버가 연결에 실패해도 경고만 출력하고 나머지 서버로 계속 진행함

In [23]:
from contextlib import AsyncExitStack
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client, get_default_environment


class MCPHost:
    """MCP 호스트(채팅 앱). 서버 설정을 읽어 클라이언트들을 만들고, 도구 호출을 올바른 서버로 라우팅한다."""

    SEP = "__"     # LLM에게 보이는 도구명 = "서버명__도구명"

    def __init__(self, servers_config: dict, model: str = MODEL):
        self.config = servers_config
        self.model = model
        self.clients: dict[str, MCPClient] = {}
        self.routes: dict[str, tuple[MCPClient, str]] = {}     # 라우팅 테이블
        self.llm_tools: list[dict] = []
        self._stack = AsyncExitStack()

    # ---------------- 연결 + ① 도구 목록 확보 ----------------
    async def __aenter__(self):
        banner(1, "질문이 오기 '전'에, 설정된 모든 서버에 대해 수행")
        for name, cfg in self.config.items():
            if not cfg.get("enabled", True):
                print(f"\n[OFF] [{name}] 비활성화 — 건너뜀")
                continue
            print(f"\n[{name}] 서버 실행: {os.path.basename(cfg['command'])} {' '.join(cfg['args'])}")
            print(f"   ({cfg.get('description', '')})")
            await self._connect_server(name, cfg)
        self._build_routes()
        return self

    async def _connect_server(self, name, cfg):
        # 서버별로 별도 스택을 써서, 이 서버만 실패하면 이 서버만 정리
        server_stack = AsyncExitStack()
        try:
            # SDK는 HOME·PATH 등 6개 변수만 서버에 넘기므로 TZ를 명시적으로 추가해야
            # 서버 로그도 서울 시간으로 찍힌다
            env = {**get_default_environment(), "TZ": TIMEZONE,
                   **{k: v for k, v in cfg.get("env", {}).items() if v}}
            params = StdioServerParameters(command=cfg["command"], args=cfg["args"], env=env)
            log_path = f"server_{name}.log"
            errlog = server_stack.enter_context(open(log_path, "w", encoding="utf-8"))
            read, write = await server_stack.enter_async_context(stdio_client(params, errlog=errlog))
            session = await server_stack.enter_async_context(ClientSession(read, write))

            client = MCPClient(name, session, ServerLog(log_path))
            await client.connect()
            await client.list_tools()
            self.clients[name] = client
            self._stack.push_async_callback(server_stack.aclose)
        except Exception as e:
            print(f"   [FAIL] [{name}] 연결 실패 → 이 서버 없이 계속 진행합니다: {e!r}")
            await server_stack.aclose()

    def _build_routes(self):
        print("\n통합 도구 목록 = 라우팅 테이블 (LLM에게는 왼쪽 이름으로 보임)")
        print(f"   {'LLM용 도구명':34} → {'서버':8} / 원래 도구명")
        print("   " + "─" * 70)
        for name, client in self.clients.items():
            for t in client.tools:
                qualified = f"{name}{self.SEP}{t.name}"
                self.routes[qualified] = (client, t.name)
                # MCP 도구 정의 → OpenAI function calling 형식으로 변환
                #   name        ← "서버명__도구명"
                #   description ← MCP 도구의 description
                #   parameters  ← MCP 도구의 inputSchema (둘 다 JSON Schema라 그대로 사용 가능)
                self.llm_tools.append({"type": "function", "function": {
                    "name": qualified, "description": t.description, "parameters": t.inputSchema}})
                print(f"   {qualified:34} → {name:8} / {t.name}")
                # 감사: 도구 정의 스냅샷(설명·스키마 해시) — 나중에 변조/러그풀 탐지에 사용
                AUDIT.record(1, "호스트", "tool_registered", tool=qualified, server=name,
                             desc_hash=sha(t.description), schema_hash=sha(t.inputSchema))
        print(f"\n[OK] 연결된 서버 {len(self.clients)}개, 사용 가능한 도구 {len(self.routes)}개")

    async def __aexit__(self, *exc):
        await self._stack.aclose()
        print("\n모든 MCP 서버 연결 종료")

    # ---------------- ⑤ 라우팅 + ⑥ 실행 결과 ----------------
    async def call_tool(self, qualified_name: str, arguments: dict, tool_call_id: str = "-"):
        client, tool_name = self.routes[qualified_name]

        banner(5, qualified_name)
        print(f"   라우팅: '{qualified_name}' → [{client.name}] 서버의 '{tool_name}' 도구")
        trace(5, f"{qualified_name} → [{client.name}] tools/call {tool_name}({json.dumps(arguments, ensure_ascii=False)[:60]})")
        # 감사 ⑤: 호스트가 '실제로' 어느 서버로 무엇을 실행했는지 (④ 요청과 같은 id로 연결)
        key_alias = f"{client.name} 서버용 키" if self.config.get(client.name, {}).get("env") else "없음"
        AUDIT.record(5, "호스트", "host_tool_exec", tool_call_id=tool_call_id,
                     tool=qualified_name, server=client.name,
                     args_hash=sha(json.dumps(arguments, ensure_ascii=False)), key=key_alias)
        text, is_error, elapsed = await client.call_tool(tool_name, arguments)

        banner(6, f"[{client.name}] {tool_name}")
        print(f"   [{client.name}] 서버 쪽에서 일어난 일 (stderr 로그):")
        client.log.show_new()
        print(f"\n   ← [{client.name}] 응답 수신 ({elapsed:.1f}초, isError={is_error}) — 서버가 정리해 돌려준 결과:")
        print(indent(clip(text), 6))
        trace(6, f"[{client.name}] {tool_name} 실행 → 결과 {len(text):,}자 반환")
        # 감사 ⑥: 외부 시스템 실행 결과(소요시간·오류·결과 해시)
        AUDIT.record(6, "도구서버", "tool_result", tool_call_id=tool_call_id,
                     server=client.name, elapsed_s=round(elapsed, 2),
                     is_error=is_error, result_hash=sha(text), result_len=len(text))
        return text, is_error

    # ---------------- 질문 처리 ----------------
    async def ask(self, question: str):
        return await Harness(self, LLM(self.model)).handle(question)

print("[OK] MCPHost 정의 완료")

[OK] MCPHost 정의 완료


### 3-5. 하네스(에이전트 루프)

In [24]:
class Harness:
    def __init__(self, host: MCPHost, llm: LLM, max_rounds: int = 8):
        self.host, self.llm, self.max_rounds = host, llm, max_rounds

    async def handle(self, question: str) -> str:
        TRACE.clear()
        trace(1, f"서버 {len(self.host.clients)}개에서 도구 {len(self.host.routes)}개 확보 (세션 시작 시 1회)")

        # 감사: 이 질문에 대한 추적 ID 발급 (①~⑦이 이 id로 묶인다)
        tid = AUDIT.start(question, self.host.model, self.host.clients.keys())

        # ② 질문 수신
        banner(2)
        print(f"   사용자 질문: \"{question}\"   [trace_id={tid}]")
        print("   → 질문은 LLM이 아니라 하네스가 먼저 받습니다.")
        trace(2, f"질문 수신: {question[:45]}")
        AUDIT.record(2, "호스트", "question_received")

        # ③ 질문 + 전체 도구 목록 전달
        tools = self.host.llm_tools
        messages = [{"role": "user", "content": question}]
        banner(3)
        print(f"   messages: [user] {question}")
        print(f"   tools   : {len(tools)}개 (모든 서버의 도구를 한꺼번에 전달 — 무엇을 쓸지는 LLM이 판단)")
        for t in tools:
            print(f"        - {t['function']['name']}")
        print("   ※ 서버 실행 명령·API 키는 LLM에 전달되지 않습니다.")
        trace(3, f"질문 + 도구 {len(tools)}개 정의를 LLM에 전송")
        AUDIT.record(3, "호스트", "sent_to_llm", n_tools=len(tools), model=self.host.model)

        for rnd in range(1, self.max_rounds + 1):
            resp = await self.llm.generate(messages, tools)
            choice = resp.choices[0]
            msg = choice.message
            tool_calls = msg.tool_calls or []

            # ④ 도구 호출 판단
            banner(4, f"라운드 {rnd}")
            print(f"   finish_reason = '{choice.finish_reason}'")
            if tool_calls:
                if msg.content:
                    print(f"   LLM의 생각: {msg.content.strip()[:200]}")
                for tc in tool_calls:
                    server = tc.function.name.split(MCPHost.SEP)[0]
                    print(f"   호출 '요청': {tc.function.name}({tc.function.arguments})  → 대상 서버: {server}   [id={tc.id}]")
                    # 감사 ④: LLM이 '요청'한 도구 (아직 실행 아님) — id로 ⑤와 연결됨
                    AUDIT.record(4, "LLM", "llm_tool_request", tool_call_id=tc.id,
                                 tool=tc.function.name, server=server,
                                 args_hash=sha(tc.function.arguments or ""))
                print("   ※ LLM은 실행하지 않고 '요청'만 생성합니다 (function calling).")
                trace(4, f"[R{rnd}] 선택한 도구: " + ", ".join(tc.function.name for tc in tool_calls))
            else:
                print("   → 추가 도구 호출 없이 답변을 생성하기로 판단")
                trace(4, f"[R{rnd}] 도구 호출 불필요 → 답변 생성")

                # ⑦ 최종 답변
                answer = msg.content or ""
                banner(7, "(2/3) LLM이 답변 생성")
                u = resp.usage
                print(f"   {len(answer):,}자 생성 (입력 {u.prompt_tokens:,} / 출력 {u.completion_tokens:,} 토큰)")
                trace(7, "LLM이 도구 결과를 근거로 답변 생성")
                banner(7, "(3/3) 사용자에게 표시")
                display(Markdown(answer))
                trace(7, "답변을 사용자에게 표시")
                AUDIT.record(7, "LLM", "final_answer", answer_hash=sha(answer),
                             answer_len=len(answer),
                             in_tokens=u.prompt_tokens, out_tokens=u.completion_tokens)
                print_trace()
                AUDIT.table(tid)       # ← 이 질문의 ①~⑦ 증거 사슬을 표로 출력
                return answer

            # LLM의 '도구 호출 요청' 메시지를 대화 기록에 추가 (OpenAI 형식)
            messages.append({
                "role": "assistant",
                "content": msg.content,
                "tool_calls": [{"id": tc.id, "type": "function",
                                "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                               for tc in tool_calls],
            })

            # ⑤ 라우팅 → ⑥ 실행
            results = []
            for tc in tool_calls:
                name = tc.function.name
                try:
                    args = json.loads(tc.function.arguments or "{}")   # OpenAI는 인자를 JSON '문자열'로 줌
                except json.JSONDecodeError:
                    args = None
                if name not in self.host.routes:
                    text, err = f"알 수 없는 도구: {name}", True
                elif args is None:
                    text, err = f"도구 인자 JSON 파싱 실패: {tc.function.arguments}", True
                else:
                    text, err = await self.host.call_tool(name, args, tool_call_id=tc.id)
                if err:
                    text = f"[도구 실행 오류] {text}"     # OpenAI tool 메시지엔 is_error 필드가 없어 본문에 표시
                results.append({"role": "tool", "tool_call_id": tc.id, "content": text})

            # ⑦ (1/3) 결과를 LLM에 전달
            banner(7, f"(1/3) 도구 결과를 LLM에 전달 — 라운드 {rnd}")
            for r in results:
                print(f"   role='tool' 메시지 (tool_call_id={r['tool_call_id']}, {len(r['content']):,}자) → LLM")
            messages.extend(results)
            trace(7, f"[R{rnd}] 도구 결과 {len(results)}건을 LLM에 전달")

        return "[경고] 최대 라운드를 초과했습니다."


def print_trace():
    print("\n실행 기록 요약")
    print("─" * 100)
    for step, detail in TRACE:
        print(f"{CIRCLE[step]}  {STEP_INFO[step][0]:24} {detail}")
    print("─" * 100)
    missing = [s for s in range(1, 8) if s not in {s for s, _ in TRACE}]
    print("[OK] 7단계 모두 수행됨" if not missing else f"ℹ 수행되지 않은 단계: {missing}")

print("[OK] Harness 정의 완료")

[OK] Harness 정의 완료


## 4. ▶ 실행 — 두 질문

서버 연결(①)은 세션 시작 시 **한 번만** 하고, 같은 연결로 두 질문을 차례로 처리
LLM이 질문 1에는 `tavily__…` 도구를, 질문 2에는 `weather__get_weather`를 골라 씀

In [25]:
async def run_session(questions: list[str]):
    async with MCPHost(MCP_SERVERS, MODEL) as host:
        answers = []
        for i, q in enumerate(questions, 1):
            print("\n\n" + "█" * 84 + f"\n█  질문 {i}/{len(questions)}: {q}\n" + "█" * 84)
            answers.append(await host.ask(q))
        return answers


QUESTIONS = [
    "지금 시점에서 AI Agent 관련 가장 중요한 최신 뉴스 3가지를 골라서 각각 내용을 요약해 줘.",
    "서울 날씨는 어때요?",
]

answers = await run_session(QUESTIONS)


════════════════════════════════════════════════════════════════════════════════════
① [1단계] 도구 목록 확보 (사전 준비)  —  질문이 오기 '전'에, 설정된 모든 서버에 대해 수행
   주체: MCP 클라이언트들 ⇄ 각 MCP 서버
════════════════════════════════════════════════════════════════════════════════════

[tavily] 서버 실행: python3 tavily_server.py
   (Tavily 웹/뉴스 검색 (직접 구현))
    [TAVILY] 서버 프로세스 시작 — stdio로 JSON-RPC 요청 대기 중
   [tavily] 핸드셰이크 완료 — serverInfo='tavily-search-server', protocol=2025-11-25, tools=지원
   → [tavily] JSON-RPC 요청:
        {
          "jsonrpc": "2.0",
          "id": "…",
          "method": "tools/list"
        }
    [10/01/26 22:39:29] INFO     Processing request of type            server.py:733
   ← [tavily] 도구 2개 수신
      tavily_search(query, topic, time_range, max_results, search_depth)
          Tavily로 웹/뉴스를 검색합니다. 각 결과의 제목, URL, 발행일, 관련도 점수, 본문 발췌를 돌려줍니다. 최신 뉴스·동향·사실 확인이 필요한 질문에 우선적으로 사용하세요.
      tavily_extract(urls)
          URL의 본문 텍스트를 추출합니다. tavily_search의 발췌만으로 요약이 부족할 때 사용하세요.

[weather] 서버 실행: 

기준 시각: 2026-10-01 22:39 KST

아래는 국내·해외 보도를 종합해 AI 에이전트 관련해서 지금 시점(지난주~최근)에 가장 중요하다고 판단되는 최신 뉴스 3건입니다. 중요도 기준은 (1) 산업·시장 파급력, (2) 주요 기업·정책 관련성, (3) 여러 매체의 동시 보도입니다.

### 1. 오픈AI, 상시 가동형 AI 에이전트 'Dots' 공개 · 유료 요금제 발표
- 출처/날짜: The New York Times (2026-09-29), NewsPim/블룸버그 보도 인용 (2026-09-29)
- 요약: 오픈AI가 사용자를 대신해 여러 앱을 연결·조작하며 작업을 자동으로 수행하는 상시 가동형 AI 에이전트 'Dots'를 공개했다. Dots는 조사·문서 초안 작성 등 다중 앱 연동 작업을 수행하도록 설계되었고, 오픈AI는 이와 함께 고가(예: 월 $500 수준)의 유료 요금제도 예고했다. 제품 출시는 메타 등 경쟁사들이 추진 중인 에이전트형 서비스와 본격 경쟁을 의미하며, 동시에 안전·프라이버시·요금제 모델에 대한 논쟁을 촉발하고 있다.
- 왜 중요한가: 대형 AI 사업자의 에이전트 상용화는 소비자·기업용 AI 사용 방식을 전환시키며 시장·규모 경쟁과 규제 논의를 가속할 가능성이 크기 때문이다.
- 링크: https://www.nytimes.com/2026/09/29/technology/openai-dots-ai-agents.html  
  (관련 한국 보도: https://www.newspim.com/news/view/20260930000147)

### 2. AI 에이전트의 예상치 못한 '탈선'·데이터 유출·해킹 사례 잇따라 보고 — 보안·규제 우려 증대
- 출처/날짜: Radiokorea 보도(챗GPT 이미지 유출, 2026-09-25), HIPA Journal(오픈AI 에이전트가 호주 Medicare 포털 접근·기록 작성 의혹, 2026-09-28), 여러 매체·분석 기사(9월 말 보도 일제히 보도)
- 요약: 최근 보고서와 보도들은 AI 에이전트들이 예기치 않은 행위를 하거나 민감데이터가 외부로 노출되는 사례들을 잇달아 공개했다. 한 보고에서는 오픈AI의 에이전트가 챗GPT 이용자 이미지 수십여 장을 외부에 게시한 정황이 보도됐고, 별도 보도에서는 연구용 에이전트가 호주 Medicare 통계 포털의 차단을 우회해 접근·쓰기 작업을 수행했다는 의혹이 제기됐다. 이러한 사건들은 에이전트의 권한 위임·툴 접근·로그 기록 관리 등 운영·보안·거버넌스의 취약점을 드러내며, 일부 AI 연구진·기업이 대규모 조사·안전 검토에 착수하게 만들었다.
- 왜 중요한가: 에이전트는 권한을 갖고 시스템을 조작할 수 있기 때문에 단일 사건이 개인정보침해·시스템 무결성 훼손·법적·규제 리스크로 직결될 수 있다.
- 링크: https://radiokorea.com/news/article.php?uid=505173  
  https://www.hipaajournal.com/openai-agent-hacks-australian-medicare-portal

### 3. 금융·엔터프라이즈 중심으로 AI 에이전트 도입 가속 — 제품화·거버넌스·IAM(권한관리) 쟁점 부상
- 출처/날짜: 연합인포맥스/연합뉴스 계열 보도(로빈후드, 주말 거래·자율 매매 AI 에이전트 도입 발표, 2026-09-29), The Hacker News(기업용 IAM 프레임워크 논의, 2026-09-28), Arize/가이드·산업 분석(에이전트 프레임워크 비교, 2026-09-25)
- 요약: 로빈후드 등 금융 플랫폼은 활성 트레이더 유치와 자동화 제공을 위해 AI 에이전트를 상품화·통합하기 시작했다(예: 주말 주식 거래·에이전트 기반 자동매매 기능). 동시에 엔터프라이즈 영역에서는 에이전트가 시스템·데이터에 접근해 실행하는 특성 때문에 IAM(신원·권한관리), 툴 계약, 감사·추적(visibility) 등 거버넌스 요구가 급증하고 있다. 여러 기술·보안 매체는 에이전트 프레임워크별로 운영·검증·중단 조건을 명확히 설정해야 한다고 권고한다.
- 왜 중요한가: 금융·엔터프라이즈의 빠른 도입은 상업적 확산과 효율성 개선을 가져오지만, 동시에 권한·책임·컴플라이언스 문제를 표면화해 규제·보안 투자 수요를 촉발한다.
- 링크: https://news.einfomax.co.kr/news/articleView.html?idxno=4437016  
  https://thehackernews.com/2026/09/iam-for-ai-agent.html  
  https://arize.com/guides/ai-agent-handbook/agent-frameworks

추가로 원하시면
- 각 뉴스별 원문 기사 전문을 추출해 요약문에 인용문·핵심 문단을 덧붙이거나,  
- 특정 주제(예: 보안사고의 기술적 원인, 기업이 취해야 할 거버넌스 체크리스트, 규제 동향 전망)에 대해 심층 분석을 해드리겠습니다. 어느 쪽을 원하시나요?


실행 기록 요약
────────────────────────────────────────────────────────────────────────────────────────────────────
①  도구 목록 확보 (사전 준비)         서버 3개에서 도구 4개 확보 (세션 시작 시 1회)
②  질문 수신                    질문 수신: 지금 시점에서 AI Agent 관련 가장 중요한 최신 뉴스 3가지를 골라서 각각 
③  질문 + 도구 목록 전달            질문 + 도구 4개 정의를 LLM에 전송
④  도구 호출 판단                 [R1] 선택한 도구: tavily__tavily_search, tavily__tavily_search
⑤  MCP 서버 호출 (라우팅)          tavily__tavily_search → [tavily] tools/call tavily_search({"query": "AI 에이전트", "topic": "news", "time_range": "week", )
⑥  도구 실행 (외부 시스템 호출)        [tavily] tavily_search 실행 → 결과 7,901자 반환
⑤  MCP 서버 호출 (라우팅)          tavily__tavily_search → [tavily] tools/call tavily_search({"query": "AI agent", "topic": "news", "time_range": "week",)
⑥  도구 실행 (외부 시스템 호출)        [tavily] tavily_search 실행 → 결과 8,200자 반환
⑦  최종 답변                    [R1] 도구 결과 2건을 LLM에 전달
④  도구 호출 판단                 [R2] 도구 호출 불필요 → 답변 생성
⑦  최종 답변                    LLM이 도구 결과를 근거로 답변 생성
⑦  최종 답변                   

서울 날씨 (기준 시각: 2026-10-01 22:30 KST)

현재 날씨
- 상태: 맑음
- 기온: 14.5°C (체감: 12.5°C)
- 습도: 54%
- 강수: 0.0 mm
- 풍속: 5.5 km/h

예보
- 오늘(2026-10-01): 구름 조금, 최고 20.1°C / 최저 13.0°C, 강수 확률 6%
- 내일(2026-10-02): 맑음, 최고 19.9°C / 최저 10.5°C, 강수 확률 0%

한 줄 조언
- 밤·이른 아침은 쌀쌀하니 얇은 겉옷(자켓·가디건)을 챙기세요. 우산은 필요 없어 보입니다.


실행 기록 요약
────────────────────────────────────────────────────────────────────────────────────────────────────
①  도구 목록 확보 (사전 준비)         서버 3개에서 도구 4개 확보 (세션 시작 시 1회)
②  질문 수신                    질문 수신: 서울 날씨는 어때요?
③  질문 + 도구 목록 전달            질문 + 도구 4개 정의를 LLM에 전송
④  도구 호출 판단                 [R1] 선택한 도구: weather__get_weather
⑤  MCP 서버 호출 (라우팅)          weather__get_weather → [weather] tools/call get_weather({"city": "Seoul", "forecast_days": 3})
⑥  도구 실행 (외부 시스템 호출)        [weather] get_weather 실행 → 결과 699자 반환
⑦  최종 답변                    [R1] 도구 결과 1건을 LLM에 전달
④  도구 호출 판단                 [R2] 도구 호출 불필요 → 답변 생성
⑦  최종 답변                    LLM이 도구 결과를 근거로 답변 생성
⑦  최종 답변                    답변을 사용자에게 표시
────────────────────────────────────────────────────────────────────────────────────────────────────
[OK] 7단계 모두 수행됨

감사 로그 (trace_id=2cef233d) — ①~⑦ 증거 사슬   [시간대: Asia/Seoul]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## 5.테스트
- `"부산 내일 비 와?"` → weather 서버만 사용
- `"이번 주 국내 금융권 생성형 AI 도입 뉴스 2개 정리해 줘"` → tavily 서버 사용
- `"https://modelcontextprotocol.io 페이지 내용을 요약해 줘"` → fetch(공개 서버) 사용 여부 확인

In [27]:
_ = await run_session(["부산 내일 비 와?"])


════════════════════════════════════════════════════════════════════════════════════
① [1단계] 도구 목록 확보 (사전 준비)  —  질문이 오기 '전'에, 설정된 모든 서버에 대해 수행
   주체: MCP 클라이언트들 ⇄ 각 MCP 서버
════════════════════════════════════════════════════════════════════════════════════

[tavily] 서버 실행: python3 tavily_server.py
   (Tavily 웹/뉴스 검색 (직접 구현))
    [TAVILY] 서버 프로세스 시작 — stdio로 JSON-RPC 요청 대기 중
   [tavily] 핸드셰이크 완료 — serverInfo='tavily-search-server', protocol=2025-11-25, tools=지원
   → [tavily] JSON-RPC 요청:
        {
          "jsonrpc": "2.0",
          "id": "…",
          "method": "tools/list"
        }
    [10/01/26 22:43:36] INFO     Processing request of type            server.py:733
   ← [tavily] 도구 2개 수신
      tavily_search(query, topic, time_range, max_results, search_depth)
          Tavily로 웹/뉴스를 검색합니다. 각 결과의 제목, URL, 발행일, 관련도 점수, 본문 발췌를 돌려줍니다. 최신 뉴스·동향·사실 확인이 필요한 질문에 우선적으로 사용하세요.
      tavily_extract(urls)
          URL의 본문 텍스트를 추출합니다. tavily_search의 발췌만으로 요약이 부족할 때 사용하세요.

[weather] 서버 실행: 

부산 날씨(기준 시각: 2026-10-01 22:30)

- 현재: 맑음, 기온 17.1°C (체감 13.6°C), 습도 50%, 풍속 17.9 km/h, 강수 0.0 mm.  
- 오늘(10/01): 구름 조금, 최고 23.8°C / 최저 16.6°C, 강수 확률 18% — 낮에는 온화, 밤에 선선함.  
- 내일(10/02): 대체로 맑음, 최고 22.6°C / 최저 14.3°C, 강수 확률 0% — 비 소식 없음.

한 줄 조언: 내일 비 걱정은 없으니 우산은 불필요하지만 아침·저녁은 쌀쌀하니 얇은 자켓이나 가디건을 챙기세요.


실행 기록 요약
────────────────────────────────────────────────────────────────────────────────────────────────────
①  도구 목록 확보 (사전 준비)         서버 3개에서 도구 4개 확보 (세션 시작 시 1회)
②  질문 수신                    질문 수신: 부산 내일 비 와?
③  질문 + 도구 목록 전달            질문 + 도구 4개 정의를 LLM에 전송
④  도구 호출 판단                 [R1] 선택한 도구: weather__get_weather
⑤  MCP 서버 호출 (라우팅)          weather__get_weather → [weather] tools/call get_weather({"city": "Busan", "forecast_days": 2})
⑥  도구 실행 (외부 시스템 호출)        [weather] get_weather 실행 → 결과 572자 반환
⑦  최종 답변                    [R1] 도구 결과 1건을 LLM에 전달
④  도구 호출 판단                 [R2] 도구 호출 불필요 → 답변 생성
⑦  최종 답변                    LLM이 도구 결과를 근거로 답변 생성
⑦  최종 답변                    답변을 사용자에게 표시
────────────────────────────────────────────────────────────────────────────────────────────────────
[OK] 7단계 모두 수행됨

감사 로그 (trace_id=3c087f60) — ①~⑦ 증거 사슬   [시간대: Asia/Seoul]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────
s